# Setup

In [ ]:
!git clone ANONYMIZED_REPO_URL (I will make this public this later)

Cloning into 'RL_Signaling'...
remote: Enumerating objects: 1081, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 1081 (delta 67), reused 117 (delta 19), pack-reused 910 (from 2)
Receiving objects: 100% (1081/1081), 116.61 MiB | 47.78 MiB/s, done.
Resolving deltas: 100% (592/592), done.


In [11]:
%cd RL_Signaling

/content/RL_Signaling/RL_Signaling


In [12]:
import os
import random

import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm

from agents import UrnAgent, QLearningAgent, TDLearningAgent
from environment import NetMultiAgentEnv, TempNetMultiAgentEnv
from simulation_function import simulation_function, temp_simulation_function
from utils import create_random_canonical_game

from joblib import Parallel, delayed, cpu_count
import multiprocessing
from datetime import datetime
import os

In [13]:
# Decide where to put the files and do the working
from google.colab import drive
drive.mount('/content/drive')

dump_path = '/content/drive/My Drive/Colab Projects/Python ABMs/Communication/Plots and Datasets/'
print("Current Directory:", dump_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Directory: /content/drive/My Drive/Colab Projects/Python ABMs/Communication/Plots and Datasets/


# Canonical Model with Signal Costs

- World States: Two binary variables X, Y
- agents_observed_variables = {0:[0],1:[1]}
- Random Canonical Games
- n_features = 2
- n_signaling_actions = 2
- n_final_actions = 4
- Signal Cost in [0.1,0.5]

In [14]:
# add_data = False
# n_iterations = 10

## Urn Agent

In [15]:
simulate = False
if simulate:
    add_data = False
    n_iterations = 10000

    # Report available CPU cores
    try:
        n_cores = cpu_count()
    except NotImplementedError:
        n_cores = 1
    print(f"Using all available CPU cores: {n_cores}")

    # Define column names
    column_names = [
        'iteration', 'n_signaling_actions', 'n_final_actions', 'full_information', 'with_signals', 'Signal_Cost_A0', 'Signal_Cost_A1',
        'Agent_0_Initial_NMI', 'Agent_0_NMI', 'Agent_0_avg_reward', 'Agent_0_final_reward',
        'Agent_1_Initial_NMI', 'Agent_1_NMI', 'Agent_1_avg_reward', 'Agent_1_final_reward'
    ]

    # Simulation parameters
    n_episodes = 10000
    n_agents = 2
    n_features = 2
    n_signaling_actions = 2
    n_final_actions = 4

    def run_single_case(iteration, full_information, with_signals, signal_cost, game_dicts, obs_vars, graph):
        # set seeds to ensure reproducibility for this specific iteration case
        np.random.seed(iteration)
        random.seed(iteration)

        env = NetMultiAgentEnv(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            full_information=full_information,
            game_dicts=game_dicts,
            observed_variables=obs_vars,
            agent_type=UrnAgent,
            initialize=False,
            costly_signaling=True,
            graph=graph
        )

        results = [iteration, n_signaling_actions, n_final_actions, full_information, with_signals, signal_cost[0], signal_cost[1]]

        # Run simulation
        signal_usage, rewards_history, signal_information_history, nature_history, histories = simulation_function(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            n_episodes=n_episodes,
            with_signals=with_signals,
            plot=False,  # Typically False for parallel runs to avoid GUI issues
            env=env,
            signal_cost=signal_cost,
            costly_signaling=True,
            verbose=False
        )

        for agent_id in range(n_agents):
            info_hist = signal_information_history[agent_id]
            reward_hist = rewards_history[agent_id]
            results.extend([
                np.mean(info_hist[:10]) if len(info_hist) > 0 else 0,     # Agent_X_Initial_NMI
                np.mean(info_hist[-100:]) if len(info_hist) > 0 else 0,   # Agent_X_NMI
                np.mean(reward_hist),                                     # Agent_X_avg_reward
                np.mean(reward_hist[-100:])                               # Agent_X_final_reward
            ])
        return results

    def run_all_cases_for_iteration(iteration):
        # Create games for this iteration
        game_dicts = {i: create_random_canonical_game(n_features, n_final_actions, n=1, m=0) for i in range(n_agents)}
        obs_vars = {0: [0], 1: [1]}

        rdn = np.random.uniform(0.0, 0.5)
        signal_cost = [rdn, rdn]

        G = nx.DiGraph()
        G.add_edges_from([(0, 1), (1, 0)])

        cases = [(False, True)]
        return [run_single_case(iteration, fi, ws, signal_cost, game_dicts, obs_vars, G) for fi, ws in cases]

    # Modified function using FixedUrnAgent logic (manually instantiated agents)
    def run_single_case_fixed(iteration, full_information, with_signals, signal_cost, game_dicts, obs_vars, graph):
        np.random.seed(iteration)
        random.seed(iteration)

        env = NetMultiAgentEnv(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            full_information=full_information,
            game_dicts=game_dicts,
            observed_variables=obs_vars,
            initialize=False,
            costly_signaling=True,
            graph=graph
        )

        # Manually instantiate agents
        env.agents = [
            UrnAgent(
                n_observed_features=n_features,
                n_signaling_actions=n_signaling_actions,
                n_final_actions=n_final_actions,
                costly_signaling=True
            ) for _ in range(n_agents)
        ]

        results = [iteration, n_signaling_actions, n_final_actions, full_information, with_signals, signal_cost[0], signal_cost[1]]

        signal_usage, rewards_history, signal_information_history, nature_history, histories = simulation_function(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            n_episodes=n_episodes,
            with_signals=with_signals,
            plot=False,
            env=env,
            signal_cost=signal_cost,
            costly_signaling=True,
            verbose=False
        )

        for agent_id in range(n_agents):
            info_hist = signal_information_history[agent_id]
            reward_hist = rewards_history[agent_id]
            results.extend([
                np.mean(info_hist[:10]) if len(info_hist) > 0 else 0,
                np.mean(info_hist[-100:]) if len(info_hist) > 0 else 0,
                np.mean(reward_hist),
                np.mean(reward_hist[-100:])
            ])
        return results

    def run_all_cases_for_iteration_fixed(iteration):
        game_dicts = {i: create_random_canonical_game(n_features, n_final_actions, n=1, m=0) for i in range(n_agents)}
        obs_vars = {0: [0], 1: [1]}

        rdn = np.random.uniform(0.0, 0.5)
        signal_cost = [rdn, rdn]

        G = nx.DiGraph()
        G.add_edges_from([(0, 1), (1, 0)])

        cases = [(False, True)]
        # Note: calling run_single_case_fixed here
        return [run_single_case_fixed(iteration, fi, ws, signal_cost, game_dicts, obs_vars, G) for fi, ws in cases]

    if __name__ == "__main__":
        # Parallel execution
        # Using n_jobs=n_cores to maximize utility
        all_results = Parallel(n_jobs=n_cores)(
            delayed(run_all_cases_for_iteration_fixed)(i) for i in tqdm(range(n_iterations), desc="Running UrnAgent simulations")
        )

        # Flatten results and create DataFrame
        flat_results = [row for group in all_results for row in group]
        results_df = pd.DataFrame(flat_results, columns=column_names)

        # Append or save
        output_file = os.path.join(dump_path, 'urnagent_results_canonical_costly_signal.csv')

        if add_data and os.path.exists(output_file):
            old_results_df = pd.read_csv(output_file)
            total_results_df = pd.concat([old_results_df, results_df], ignore_index=True)
        else:
            total_results_df = results_df

        total_results_df.to_csv(output_file, index=False)
        print(f"Total rows in saved file: {len(total_results_df)}")

In [16]:
# simulate=True
# if simulate:
#   add_data = False
#   n_iterations = 10000

#   # Report available CPU cores
#   n_cores = cpu_count()
#   print(f"Using all available CPU cores: {n_cores}")

#   # Define column names
#   column_names = [
#       'iteration', 'n_signaling_actions', 'n_final_actions', 'full_information', 'with_signals','Signal_Cost_A0', 'Signal_Cost_A1',
#       'Agent_0_Initial_NMI', 'Agent_0_NMI', 'Agent_0_avg_reward', 'Agent_0_final_reward',
#       'Agent_1_Initial_NMI', 'Agent_1_NMI', 'Agent_1_avg_reward', 'Agent_1_final_reward'
#   ]

#   # Simulation parameters
#   n_episodes = 10000
#   n_agents = 2
#   n_features = 2
#   n_signaling_actions = 2
#   n_final_actions = 4

#   def run_single_case(iteration, full_information, with_signals, signal_cost,game_dicts, obs_vars, graph):
#       #set seeds
#       np.random.seed(iteration)
#       random.seed(iteration)
#       # continue

#       env = NetMultiAgentEnv(n_agents=n_agents, n_features=n_features,
#                   n_signaling_actions=n_signaling_actions,
#                   n_final_actions=n_final_actions,
#                   full_information = full_information,
#                   game_dicts=game_dicts,
#                   observed_variables = obs_vars,
#                   agent_type=UrnAgent,
#                   initialize = False,
#                   costly_signaling=True,
#                   graph=graph)

#       results = [iteration, n_signaling_actions, n_final_actions, full_information, with_signals,signal_cost[0],signal_cost[1]]

#       signal_usage, rewards_history, signal_information_history, nature_history, histories = simulation_function(n_agents=n_agents,
#                       n_features=n_features, n_signaling_actions=n_signaling_actions,
#                       n_final_actions=n_final_actions,
#                       n_episodes=n_episodes, with_signals = with_signals, plot=True,env=env,
#                       signal_cost = signal_cost,
#                       costly_signaling=True, verbose=False)

#       for agent_id in range(n_agents):
#           info_hist = signal_information_history[agent_id]
#           reward_hist = rewards_history[agent_id]
#           results.extend([
#             np.mean(info_hist[:10]),       # 'Agent_X_Initial_NMI'  — mean over first 10 episodes
#             np.mean(info_hist[-100:]),     # 'Agent_X_NMI'          — mean over last 100 episodes
#             np.mean(reward_hist),          # 'Agent_X_avg_reward'   — mean over all episodes
#             np.mean(reward_hist[-100:])    # 'Agent_X_final_reward' — mean over last 100 episodes
#             ])
#       return results

#   def run_all_cases_for_iteration(iteration):
#       # continue
#       game_dicts = {i: create_random_canonical_game(n_features, n_final_actions, n=1, m=0) for i in range(n_agents)}
#       obs_vars = {0: [0], 1: [1]}
#       rdn = np.random.uniform(0.0, 0.5)
#       signal_cost = [rdn,rdn]
#       G = nx.DiGraph()
#       G.add_edges_from([(0, 1), (1, 0)])

#       cases = [(False, True)]
#       return [run_single_case(iteration, fi, ws, signal_cost,game_dicts, obs_vars, G) for fi, ws in cases]


#   # Modify the env creation to use the FixedUrnAgent
#   def run_single_case_fixed(iteration, full_information, with_signals, signal_cost,game_dicts, obs_vars, graph):
#       np.random.seed(iteration)
#       random.seed(iteration)

#       env = NetMultiAgentEnv(n_agents=n_agents, n_features=n_features,
#                   n_signaling_actions=n_signaling_actions,
#                   n_final_actions=n_final_actions,
#                   full_information = full_information,
#                   game_dicts=game_dicts,
#                   observed_variables = obs_vars,
#                   # The agent_type parameter is likely ignored when initialize=False
#                   initialize = False, # Keep initialize False, as agents are manually instantiated below
#                   costly_signaling=True,
#                   graph=graph)

#       # Manually instantiate and assign FixedUrnAgent objects to env.agents
#       # This is consistent with the pattern used for QLearningAgent when initialize=False.
#       # The n_signaling_actions is passed directly, as the q_table indexing in get_action uses observation[1]
#       # directly as the signal_from_other_agent index.
#       env.agents = [
#           UrnAgent(
#               n_observed_features=n_features,
#               n_signaling_actions=n_signaling_actions,
#               n_final_actions=n_final_actions,
#               # choice='softmax', # Default choice for UrnAgent, used in get_action
#               # temperature=0.1, # Default temperature for UrnAgent, used in get_action
#               costly_signaling=True # Consistent with environment setup
#           ) for _ in range(n_agents)
#       ]

#       results = [iteration, n_signaling_actions, n_final_actions, full_information, with_signals,signal_cost[0],signal_cost[1]]

#       signal_usage, rewards_history, signal_information_history, nature_history, histories = simulation_function(n_agents=n_agents,
#                       n_features=n_features, n_signaling_actions=n_signaling_actions,
#                       n_final_actions=n_final_actions,
#                       n_episodes=n_episodes, with_signals = with_signals, plot=True,env=env,
#                       signal_cost = signal_cost,
#                       costly_signaling=True, verbose=False)

#       for agent_id in range(n_agents):
#           info_hist = signal_information_history[agent_id]
#           reward_hist = rewards_history[agent_id]
#           results.extend([
#             np.mean(info_hist[:10]),
#             np.mean(info_hist[-100:]),
#             np.mean(reward_hist),
#             np.mean(reward_hist[-100:])
#             ])
#       return results

#   def run_all_cases_for_iteration_fixed(iteration):
#       game_dicts = {i: create_random_canonical_game(n_features, n_final_actions, n=1, m=0) for i in range(n_agents)}
#       obs_vars = {0: [0], 1: [1]}
#       rdn = np.random.uniform(0.0, 0.5)
#       signal_cost = [rdn,rdn]
#       G = nx.DiGraph()
#       G.add_edges_from([(0, 1), (1, 0)])

#       cases = [(False, True)]
#       return [run_single_case_fixed(iteration, fi, ws, signal_cost,game_dicts, obs_vars, G) for fi, ws in cases]


#   all_results = Parallel(n_jobs=n_cores)(
#       delayed(run_all_cases_for_iteration_fixed)(i) for i in tqdm(range(n_iterations), desc="Running UrnAgent simulations")
#   )

#   # Flatten results and create DataFrame
#   flat_results = [row for group in all_results for row in group]
#   results_df = pd.DataFrame(flat_results, columns=column_names)

#   # Append or save
#   output_file = dump_path+'urnagent_results_canonical_costly_signal.csv'
#   if add_data:
#       old_results_df = pd.read_csv(output_file)
#       total_results_df = pd.concat([old_results_df, results_df], ignore_index=True)
#   else:
#       total_results_df = results_df

#   total_results_df.to_csv(output_file, index=False)
  # print(f"Total rows in saved file: {len(total_results_df)}")

## Q-Learning

In [17]:
simulate=True
if simulate:
  add_data = False
  n_iterations = 10000

  # Print number of available CPU cores
  n_cores = cpu_count()
  print(f"Using all available CPU cores: {n_cores}")

  # Define column names
  column_names = [
      'iteration', 'n_signaling_actions', 'n_final_actions', 'full_information', 'with_signals', 'Signal_Cost_A0', 'Signal_Cost_A1',
      'Agent_0_Initial_NMI', 'Agent_0_NMI', 'Agent_0_avg_reward', 'Agent_0_final_reward',
      'Agent_1_Initial_NMI', 'Agent_1_NMI', 'Agent_1_avg_reward', 'Agent_1_final_reward'
  ]

  n_episodes = 10000
  n_agents = 2
  n_features = 2
  n_signaling_actions = 2
  n_final_actions = 4

  def run_single_case(iteration, full_info, with_signals, signal_cost, game_dicts, obs_vars, graph):
          #set seeds
      np.random.seed(iteration)
      random.seed(iteration)
      # continue

      env = NetMultiAgentEnv(n_agents=n_agents, n_features=n_features,
                      n_signaling_actions=n_signaling_actions,
                      n_final_actions=n_final_actions,
                      full_information = full_info,
                      game_dicts=game_dicts,
                      observed_variables = obs_vars,
                      agent_type=QLearningAgent,
                      initialize = False,
                      costly_signaling=True,
                      graph=graph)

      effective_n_signaling_actions = n_signaling_actions + 1
      env.agents = [
              QLearningAgent(
                  n_signaling_actions=effective_n_signaling_actions,
                  n_final_actions=n_final_actions,
                  exploration_rate=0.9652628633727897,
                  exploration_decay=0.9998122815486062,
                  min_exploration_rate=1e-10,
                  choice='ucb',
                  exp_smoothing=False,
                  costly_signaling=True
              ) for _ in range(n_agents)
          ]


      results = [iteration, n_signaling_actions, n_final_actions, full_info, with_signals,signal_cost[0],signal_cost[1]]

      signal_usage, rewards_history, signal_information_history, nature_history, histories = simulation_function(n_agents=n_agents,
                      n_features=n_features, n_signaling_actions=n_signaling_actions,
                      n_final_actions=n_final_actions,
                      n_episodes=10000, with_signals = with_signals,plot=True,env=env, verbose=False,
                      signal_cost = signal_cost,
                      costly_signaling=True)

      for agent_id in range(n_agents):
          info_hist = signal_information_history[agent_id]
          reward_hist = rewards_history[agent_id]
          results.extend([
              np.mean(info_hist[:10]),
              np.mean(info_hist[-100:]),
              np.mean(reward_hist),
              np.mean(reward_hist[-100:])
          ])

      return results

  def run_all_cases_for_iteration(iteration):
      # Prepare shared data for all 4 cases
      game_dicts = {i: create_random_canonical_game(n_features, n_final_actions) for i in range(n_agents)}
      obs_vars = {0: [0], 1: [1]}
      rdn = np.random.uniform(0.0, 0.5)
      signal_cost = [rdn,rdn]
      G = nx.DiGraph()
      G.add_edges_from([(0, 1), (1, 0)])

      cases = [(False, True)]
      return [run_single_case(iteration, fi, ws, signal_cost,game_dicts, obs_vars, G) for fi, ws in cases]

  # Run simulations in parallel using all available cores
  all_results = Parallel(n_jobs=n_cores)(
      delayed(run_all_cases_for_iteration)(i) for i in tqdm(range(n_iterations), desc="Running in parallel")
  )

  # Flatten the list of lists
  flat_results = [row for group in all_results for row in group]
  results_df = pd.DataFrame(flat_results, columns=column_names)

  # Append or save
  output_file = dump_path+'qlearning_results_canonical_costly_signal.csv'
  if add_data:
      old_results_df = pd.read_csv(output_file)
      total_results_df = pd.concat([old_results_df, results_df], ignore_index=True)
  else:
      total_results_df = results_df

  total_results_df.to_csv(output_file, index=False)
  print(f"Total rows: {len(total_results_df)}")

Using all available CPU cores: 8


Running in parallel: 100%|██████████| 10000/10000 [1:53:07<00:00,  1.47it/s]


Total rows: 10000


## Disconnect from Runtime

In [18]:
from google.colab import runtime
runtime.unassign()